# Experiment 2.0 — Diagnostics

Three checks before committing 2.1 to MLP framing. The 2.0 hurdle landed at R² 0.30 against a 0.45–0.55 prior; the no-xP Lasso zeroed out every recent-form feature; and RidgeCV picked α=316 where Phase 1 ran fine at α=1. We need to know whether 2.0 underperformed because **linearity is the bottleneck** (form signal exists but lives in interactions an MLP would find) or because **signal availability is the bottleneck** (xP encodes things — team strength, FDR, set-piece taker, bookmaker-implied rates — that aren't in the raw feature set, and no model recovers what isn't there).

The three diagnostics:

1. **Stage 2 R² on played-only val, alone.** The combined hurdle R² blends Stage 1's correctly-zeroed DNPs with Stage 2's conditional estimate. Stage 2 alone is the linear ceiling on the conditional points-given-play problem — i.e. the headroom an MLP would actually compete for.
2. **Ridge α=1 vs α=316.** RidgeCV picked α=316 without xP; Phase 1's hurdle ran at α=1 with xP. If α=1 produces a meaningfully better hurdle R², RidgeCV over-corrected. If it's worse, the high-α choice was right and signal is genuinely thin.
3. **Lasso at fixed α=1e-4 (Phase 1's reference) without xP.** LassoCV picked α≈0.0074 — ~75× higher than Phase 1's reference. The original sign-flip sanity check was run under heavy regularization. Re-running at the un-regularized regime answers whether the sign-flip failure was a CV artifact or a real absence of linear signal.

Diagnostics 1 and 2 are answered by the same cell.


## Setup

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED)

DATA_PATH = Path("../data/processed/fpl_modeling_data.csv")
df = pd.read_csv(DATA_PATH)

TRAIN_SEASONS = ["2021-22", "2022-23"]
HOLDOUT_SEASON = "2023-24"
TEST_GW_START = 34

train_mask       = df["season"].isin(TRAIN_SEASONS)
val_mask         = (df["season"] == HOLDOUT_SEASON) & (df["GW"] < TEST_GW_START)
played_train_mask = train_mask & (df["played_any"] == 1)
played_val_mask   = val_mask   & (df["played_any"] == 1)

EXCLUDE = {"name", "name_key", "season", "GW", "player_season", "team", "position",
           "total_points", "played_any", "played_60min", "cluster_id"}
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
features_no_xp = [c for c in numeric_cols if c not in EXCLUDE and c != "xP"]

print(f"train: {train_mask.sum():,}  val: {val_mask.sum():,}  "
      f"played-only train: {played_train_mask.sum():,}  played-only val: {played_val_mask.sum():,}")
print(f"features (no xP): {len(features_no_xp)}")


## Diagnostics 1 & 2 — Stage 2 in isolation, α=1 vs α=316

For each alpha, report **stage-2-alone** metrics (Ridge predictions on played-only val rows, evaluated against actual `total_points`) and **combined hurdle** metrics (Stage 1 LR × Stage 2 Ridge on full val). Stage 2 alone is the headroom number that matters for 2.1.


In [ ]:
# Stage 1 LR (no xP) — same as Experiment 2.0
X_train_s1 = df.loc[train_mask, features_no_xp].values
y_train_s1 = df.loc[train_mask, "played_any"].values
X_val_s1   = df.loc[val_mask,   features_no_xp].values
y_val_s1   = df.loc[val_mask,   "played_any"].values

scaler_s1 = StandardScaler().fit(X_train_s1)
lr_s1 = LogisticRegression(max_iter=2000, random_state=SEED).fit(scaler_s1.transform(X_train_s1), y_train_s1)
p_play_no_xp = lr_s1.predict_proba(scaler_s1.transform(X_val_s1))[:, 1]

# Stage 2 setup — fit own scaler on played-only train
X_train_s2_raw       = df.loc[played_train_mask, features_no_xp].values
y_train_s2           = df.loc[played_train_mask, "total_points"].values
X_val_full_raw       = df.loc[val_mask,           features_no_xp].values
X_val_played_raw     = df.loc[played_val_mask,    features_no_xp].values

scaler_s2 = StandardScaler().fit(X_train_s2_raw)
X_train_s2     = scaler_s2.transform(X_train_s2_raw)
X_val_full     = scaler_s2.transform(X_val_full_raw)
X_val_played   = scaler_s2.transform(X_val_played_raw)

y_val_full   = df.loc[val_mask,        "total_points"].values
y_val_played = df.loc[played_val_mask, "total_points"].values

# Run Ridge at both alphas
rows = []
for alpha in [1.0, 316.2278]:
    ridge = Ridge(alpha=alpha, random_state=SEED).fit(X_train_s2, y_train_s2)

    # Stage 2 alone — predict E[points | played] on played-only val rows
    pred_played = ridge.predict(X_val_played)
    r2_s2  = r2_score(y_val_played, pred_played)
    mae_s2 = mean_absolute_error(y_val_played, pred_played)

    # Combined hurdle — P(play) * E[points | played] on full val
    cond_full = ridge.predict(X_val_full)
    pred_comb = p_play_no_xp * cond_full
    r2_c  = r2_score(y_val_full, pred_comb)
    mae_c = mean_absolute_error(y_val_full, pred_comb)

    rows.append({
        "alpha": alpha,
        "stage2_alone_R²":  round(r2_s2, 4),
        "stage2_alone_MAE": round(mae_s2, 4),
        "combined_R²":      round(r2_c, 4),
        "combined_MAE":     round(mae_c, 4),
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))

print("\nReference (from Experiment 2.0): combined R² 0.2999, combined MAE 0.9762 at α≈316")


## Diagnostic 3 — Lasso at fixed α=1e-4 (no xP)

Phase 1's reference regime. Did the sign-flip failure for `total_points_roll3` come from LassoCV's α≈0.0074 zeroing out small coefficients, or is the signal genuinely absent in linear form?


In [ ]:
# Reuse scaler_s2 — same training subset and feature set as the LassoCV run in Experiment 2.0,
# so coefficients are directly comparable in standardized units.
lasso_fixed = Lasso(alpha=1e-4, max_iter=50000, random_state=SEED).fit(X_train_s2, y_train_s2)
coef_fixed = pd.Series(lasso_fixed.coef_, index=features_no_xp)

print(f"Nonzero coefficients: {(coef_fixed != 0).sum()}/{len(coef_fixed)}")

print("\nTop 10 |coef| at fixed α=1e-4 (no xP):")
print(coef_fixed.reindex(coef_fixed.abs().sort_values(ascending=False).index)
                .head(10).round(4).to_string())

check_features = [
    "total_points_roll3", "total_points_lag1",
    "bps_roll3", "bps_lag1",
    "minutes_roll3", "minutes_lag1",
    "ict_index_roll3",
]
print("\nSign check at α=1e-4 (compare to LassoCV α≈0.0074 from Experiment 2.0):")
for f in check_features:
    val = coef_fixed.get(f, np.nan)
    sign = "POSITIVE" if val > 1e-6 else ("NEGATIVE" if val < -1e-6 else "ZERO")
    print(f"  {f:<22s} {val:+.4f}  ({sign})")


## How to read the results

**Stage 2 alone R² (Diagnostic 1)** is the relevant headroom for the MLP.

- `~0.30+` — Stage 2 already extracts most of the linear signal. MLP gain over linear is bounded; expect modest 2.1 improvement. Reframe writeup toward "xP encodes information not present in raw historical features."
- `~0.15–0.25` — Real headroom for nonlinearity. Original framing ("MLP recovers X% of xP's value") survives.
- `<~0.10` — The conditional points problem is essentially noise without xP. MLP is unlikely to help.

**Ridge α=1 vs α=316 (Diagnostic 2)**

- α=1 better → RidgeCV over-corrected; the no-xP problem isn't quite as signal-thin as it looked. Slight upward update on (a).
- α=316 better → high-α was right; signal-availability story (b) gets reinforced.
- Roughly equal → α choice doesn't matter much, which itself suggests the loss surface is flat — also consistent with (b).

**Sign of `total_points_roll3` at α=1e-4 (Diagnostic 3)**

- POSITIVE → CV artifact; Phase 1's prediction held under the un-regularized regime. Linear form has signal that LassoCV's α≈0.0074 was suppressing.
- ZERO or NEGATIVE → the signal genuinely isn't there in linear form. Form features only carried negative-coefficient information *relative to xP*; once xP is gone, they have no independent linear signal.

The cleanest reading is when all three point the same way. If they conflict, the headroom number (Diagnostic 1) is the one that matters most for the 2.1 decision.
